
## Folder 09 — File 05: Incremental Pipeline Validation:


This notebook checks whether your incremental pipeline is operating correctly across Bronze, Silver and Gold.

It validates:

- Bronze table availability and record counts.
- Silver table availability and duplicate business keys.
- Incremental processing ledger and quarantine records.
- Delta MERGE execution results.
- Gold table availability and data freshness.
- Overall pipeline validation status.


### Validation Flow


```text
+------------------------------+
| 01. BRONZE VALIDATION        |
| Tables, Metadata, Row Counts |
+------------------------------+
               |
               v
+------------------------------+
| 02. SILVER VALIDATION        |
| Business Keys, Duplicates,   |
| NULL Values                  |
+------------------------------+
               |
               v
+------------------------------+
| 03. INCREMENTAL PROCESSING   |
| Processing Ledger,           |
| Quarantine Records           |
+------------------------------+
               |
               v
+------------------------------+
| 04. DELTA MERGE VALIDATION   |
| Merge Reports, Inserts,      |
| Updates, Failures            |
+------------------------------+
               |
               v
+------------------------------+
| 05. GOLD VALIDATION          |
| Analytics Tables, KPIs,      |
| Record Counts                |
+------------------------------+
               |
               v
+------------------------------+
| 06. FINAL PIPELINE STATUS    |
| PASS / FAIL Summary          |
| Saved Validation Report     |
+------------------------------+
```

In [0]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

from datetime import datetime


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

BRONZE_DATABASE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER_DATABASE = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD_DATABASE = f"{CATALOG}.{GOLD_SCHEMA}"

VALIDATION_REPORT_TABLE = (
    f"{GOLD_DATABASE}.incremental_pipeline_validation_report"
)

LEDGER_TABLE = (
    f"{SILVER_DATABASE}.incremental_silver_processing_ledger"
)

QUARANTINE_TABLE = (
    f"{SILVER_DATABASE}.incremental_quarantine_records"
)

PROCESSING_REPORT_TABLE = (
    f"{SILVER_DATABASE}.incremental_silver_processing_report"
)

MERGE_REPORT_TABLE = (
    f"{SILVER_DATABASE}.incremental_delta_merge_audit_report"
)

GOLD_REFRESH_REPORT = (
    f"{GOLD_DATABASE}.gold_refresh_execution_report"
)


# ============================================================
# 3. DATASET CONFIGURATION
# ============================================================

DATASETS = {
    "customers": {
        "bronze_candidates": [
            f"{BRONZE_DATABASE}.customers",
            f"{BRONZE_DATABASE}.bronze_customers"
        ],
        "silver_table": f"{SILVER_DATABASE}.silver_customers",
        "primary_keys": ["customer_id"]
    },

    "products": {
        "bronze_candidates": [
            f"{BRONZE_DATABASE}.products",
            f"{BRONZE_DATABASE}.bronze_products"
        ],
        "silver_table": f"{SILVER_DATABASE}.silver_products",
        "primary_keys": ["product_id"]
    },

    "orders": {
        "bronze_candidates": [
            f"{BRONZE_DATABASE}.orders",
            f"{BRONZE_DATABASE}.bronze_orders"
        ],
        "silver_table": f"{SILVER_DATABASE}.silver_orders",
        "primary_keys": ["order_id"]
    },

    "order_items": {
        "bronze_candidates": [
            f"{BRONZE_DATABASE}.order_items",
            f"{BRONZE_DATABASE}.bronze_order_items"
        ],
        "silver_table": f"{SILVER_DATABASE}.silver_order_items",
        "primary_keys": ["order_item_id"]
    },

    "payments": {
        "bronze_candidates": [
            f"{BRONZE_DATABASE}.payments",
            f"{BRONZE_DATABASE}.bronze_payments"
        ],
        "silver_table": f"{SILVER_DATABASE}.silver_payments",
        "primary_keys": ["payment_id"]
    },

    "shipments": {
        "bronze_candidates": [
            f"{BRONZE_DATABASE}.shipments",
            f"{BRONZE_DATABASE}.bronze_shipments"
        ],
        "silver_table": f"{SILVER_DATABASE}.silver_shipments",
        "primary_keys": ["shipment_id"]
    },

    "inventory_snapshots": {
        "bronze_candidates": [
            f"{BRONZE_DATABASE}.inventory_snapshots",
            f"{BRONZE_DATABASE}.bronze_inventory_snapshots"
        ],
        "silver_table": f"{SILVER_DATABASE}.silver_inventory",
        "primary_keys": [
            "product_id",
            "warehouse_id",
            "snapshot_date"
        ]
    }
}


# ============================================================
# 4. GOLD TABLE CONFIGURATION
# ============================================================

# IMPORTANT:

# gold_order_payment_summary is the actual table name.
# Do not use gold_order_payment_reconciliation.

GOLD_TABLES = [
    "gold_daily_sales",
    "gold_channel_daily_sales",
    "gold_product_performance",
    "gold_category_performance",
    "gold_customer_lifetime_value",
    "gold_customer_segments",
    "gold_payment_metrics",
    "gold_payment_method_performance",
    "gold_order_payment_summary",
    "gold_shipping_metrics",
    "gold_inventory_status",
    "gold_warehouse_inventory",
    "gold_inventory_snapshot_trends",
    "gold_inventory_alerts",
    "gold_executive_kpis"
]


# ============================================================
# 5. VALIDATION RESULT COLLECTION
# ============================================================

validation_results = []


def add_check(
    category,
    object_name,
    check_name,
    actual_value,
    expected_value,
    passed,
    details
):

    validation_results.append({

        "validation_category": str(category),

        "object_name": str(object_name),

        "check_name": str(check_name),

        "actual_value": str(actual_value),

        "expected_value": str(expected_value),

        "validation_status": (
            "PASS" if passed else "FAIL"
        ),

        "details": str(details),

        "validation_timestamp": datetime.now()

    })


# ============================================================
# 6. HELPER FUNCTIONS
# ============================================================

def table_exists(table_name):

    try:
        return spark.catalog.tableExists(table_name)

    except Exception:
        return False


def resolve_table(candidates):

    for table_name in candidates:

        if table_exists(table_name):
            return table_name

    return None


def get_table_count(table_name):

    return spark.table(table_name).count()


def get_primary_key_columns(table_name):

    return spark.table(table_name).columns


def get_duplicate_count(table_name, primary_keys):

    df = spark.table(table_name)

    duplicate_count = (
        df.groupBy(*primary_keys)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    return duplicate_count


def get_null_key_count(table_name, primary_keys):

    df = spark.table(table_name)

    null_condition = None

    for key in primary_keys:

        condition = (
            F.col(key).isNull()
            | (F.trim(F.col(key).cast("string")) == "")
        )

        if null_condition is None:
            null_condition = condition
        else:
            null_condition = null_condition | condition

    return df.filter(null_condition).count()


def latest_batch(df):

    """
    Restrict a report to its latest timestamped batch
    when a recognized timestamp column exists.
    """

    timestamp_candidates = [
        "report_timestamp",
        "execution_timestamp",
        "processing_timestamp",
        "created_at",
        "timestamp"
    ]

    available_columns = df.columns

    for column_name in timestamp_candidates:

        if column_name in available_columns:

            latest_value = (
                df.agg(
                    F.max(F.col(column_name)).alias("latest_value")
                )
                .first()["latest_value"]
            )

            if latest_value is not None:

                filtered_df = df.filter(
                    F.col(column_name) == F.lit(latest_value)
                )

                return filtered_df, column_name, latest_value

    # No recognized timestamp column:

    # return the complete report rather than guessing a batch.

    return df, None, None


def count_failed_records(df):

    """
    Find a status column and count failed records.
    """

    possible_status_columns = [
        "execution_status",
        "processing_status",
        "validation_status",
        "status"
    ]

    status_column = None

    for column_name in possible_status_columns:

        if column_name in df.columns:
            status_column = column_name
            break

    if status_column is None:
        return None, None

    failed_count = (
        df.filter(
            F.upper(F.col(status_column).cast("string")) == "FAILED"
        )
        .count()
    )

    return failed_count, status_column


# 7. BRONZE LAYER VALIDATION

print("=" * 80)
print("BRONZE LAYER VALIDATION")
print("=" * 80)

BRONZE_METADATA_COLUMNS = [
    "_source_file",
    "_ingestion_timestamp",
    "_ingestion_date"
]

for dataset_name, config in DATASETS.items():

    print(f"\nValidating Bronze: {dataset_name}")

    bronze_table = resolve_table(
        config["bronze_candidates"]
    )

    if bronze_table is None:

        add_check(
            "BRONZE",
            dataset_name,
            "Table existence",
            "MISSING",
            "AVAILABLE",
            False,
            "Bronze table does not exist."
        )

        continue

    bronze_df = spark.table(bronze_table)

    bronze_count = bronze_df.count()

    # Check 1: Table existence

    add_check(
        "BRONZE",
        bronze_table,
        "Table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Bronze table exists."
    )

    # Check 2: Record count

    add_check(
        "BRONZE",
        bronze_table,
        "Record count",
        bronze_count,
        "> 0",
        bronze_count > 0,
        "Bronze table must contain records."
    )

    # Check 3: Ingestion metadata

    missing_metadata = [
        column
        for column in BRONZE_METADATA_COLUMNS
        if column not in bronze_df.columns
    ]

    add_check(
        "BRONZE",
        bronze_table,
        "Ingestion metadata",
        (
            "MISSING: " + ", ".join(missing_metadata)
            if missing_metadata
            else "ALL AVAILABLE"
        ),
        "ALL AVAILABLE",
        len(missing_metadata) == 0,
        "Checks source file and ingestion metadata columns."
    )

    # Check 4: Business key columns

    primary_keys = config["primary_keys"]

    missing_keys = [
        key
        for key in primary_keys
        if key not in bronze_df.columns
    ]

    add_check(
        "BRONZE",
        bronze_table,
        "Primary key columns",
        (
            "MISSING: " + ", ".join(missing_keys)
            if missing_keys
            else "ALL AVAILABLE"
        ),
        "ALL AVAILABLE",
        len(missing_keys) == 0,
        "Checks required business key columns."
    )


## 8. SILVER LAYER VALIDATION

print("=" * 80)
print("SILVER LAYER VALIDATION")
print("=" * 80)

for dataset_name, config in DATASETS.items():

    silver_table = config["silver_table"]

    primary_keys = config["primary_keys"]

    print(f"\nValidating Silver: {dataset_name}")

    if not table_exists(silver_table):

        add_check(
            "SILVER",
            silver_table,
            "Table existence",
            "MISSING",
            "AVAILABLE",
            False,
            "Silver table does not exist."
        )

        continue

    silver_df = spark.table(silver_table)

    silver_count = silver_df.count()

    # Check 1: Table existence

    add_check(
        "SILVER",
        silver_table,
        "Table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Silver table exists."
    )

    # Check 2: Record count

    add_check(
        "SILVER",
        silver_table,
        "Record count",
        silver_count,
        "> 0",
        silver_count > 0,
        "Silver table must contain records."
    )

    # Check 3: Primary key columns

    missing_keys = [
        key
        for key in primary_keys
        if key not in silver_df.columns
    ]

    add_check(
        "SILVER",
        silver_table,
        "Primary key columns",
        (
            "MISSING: " + ", ".join(missing_keys)
            if missing_keys
            else "ALL AVAILABLE"
        ),
        "ALL AVAILABLE",
        len(missing_keys) == 0,
        "Checks required Silver business key columns."
    )

    if missing_keys:
        continue

    # Check 4: Null primary keys

    null_key_count = get_null_key_count(
        silver_table,
        primary_keys
    )

    add_check(
        "SILVER",
        silver_table,
        "Null primary keys",
        null_key_count,
        0,
        null_key_count == 0,
        "Silver business keys must not be null or empty."
    )

    # Check 5: Duplicate primary keys

    duplicate_count = get_duplicate_count(
        silver_table,
        primary_keys
    )

    add_check(
        "SILVER",
        silver_table,
        "Duplicate primary keys",
        duplicate_count,
        0,
        duplicate_count == 0,
        "Checks duplicate business key groups."
    )



## 9. INCREMENTAL PROCESSING LEDGER VALIDATION

print("=" * 80)
print("INCREMENTAL LEDGER VALIDATION")
print("=" * 80)

if not table_exists(LEDGER_TABLE):

    add_check(
        "LEDGER",
        LEDGER_TABLE,
        "Table existence",
        "MISSING",
        "AVAILABLE",
        False,
        "Incremental processing ledger does not exist."
    )

else:

    ledger_df = spark.table(LEDGER_TABLE)

    ledger_count = ledger_df.count()

    add_check(
        "LEDGER",
        LEDGER_TABLE,
        "Table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Incremental ledger exists."
    )

    add_check(
        "LEDGER",
        LEDGER_TABLE,
        "Record count",
        ledger_count,
        "> 0",
        ledger_count > 0,
        "Ledger must contain processed records."
    )

    required_ledger_columns = [
        "dataset_name",
        "record_id"
    ]

    missing_columns = [
        column
        for column in required_ledger_columns
        if column not in ledger_df.columns
    ]

    add_check(
        "LEDGER",
        LEDGER_TABLE,
        "Required columns",
        (
            "MISSING: " + ", ".join(missing_columns)
            if missing_columns
            else "ALL AVAILABLE"
        ),
        "ALL AVAILABLE",
        len(missing_columns) == 0,
        "Checks dataset_name and record_id columns."
    )

    if not missing_columns:

        duplicate_ledger_count = (
            ledger_df
            .groupBy("dataset_name", "record_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        add_check(
            "LEDGER",
            LEDGER_TABLE,
            "Duplicate ledger records",
            duplicate_ledger_count,
            0,
            duplicate_ledger_count == 0,
            "Each dataset and record ID combination should be unique."
        )


# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. QUARANTINE VALIDATION

# COMMAND ----------

print("=" * 80)
print("QUARANTINE VALIDATION")
print("=" * 80)

if not table_exists(QUARANTINE_TABLE):

    add_check(
        "QUARANTINE",
        QUARANTINE_TABLE,
        "Table existence",
        "MISSING",
        "AVAILABLE",
        False,
        "Quarantine table does not exist."
    )

else:

    quarantine_df = spark.table(QUARANTINE_TABLE)

    quarantine_count = quarantine_df.count()

    add_check(
        "QUARANTINE",
        QUARANTINE_TABLE,
        "Table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Quarantine table exists."
    )

    add_check(
        "QUARANTINE",
        QUARANTINE_TABLE,
        "Record count",
        quarantine_count,
        ">= 0",
        quarantine_count >= 0,
        "Quarantined records are retained for investigation."
    )

    print("Quarantined records:", quarantine_count)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 11. DELTA MERGE AUDIT VALIDATION

# COMMAND ----------

print("=" * 80)
print("DELTA MERGE AUDIT VALIDATION")
print("=" * 80)

if not table_exists(MERGE_REPORT_TABLE):

    add_check(
        "DELTA_MERGE",
        MERGE_REPORT_TABLE,
        "Audit table existence",
        "MISSING",
        "AVAILABLE",
        False,
        "Incremental Delta MERGE audit report does not exist."
    )

else:

    merge_df = spark.table(MERGE_REPORT_TABLE)

    merge_count = merge_df.count()

    add_check(
        "DELTA_MERGE",
        MERGE_REPORT_TABLE,
        "Audit table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Delta MERGE audit report exists."
    )

    add_check(
        "DELTA_MERGE",
        MERGE_REPORT_TABLE,
        "Audit record count",
        merge_count,
        "> 0",
        merge_count > 0,
        "MERGE audit report must contain audit records."
    )

    print("MERGE audit records:", merge_count)

    print("MERGE audit columns:", merge_df.columns)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. GOLD LAYER VALIDATION

# COMMAND ----------

print("=" * 80)
print("GOLD LAYER VALIDATION")
print("=" * 80)

for gold_table_name in GOLD_TABLES:

    full_table_name = (
        f"{GOLD_DATABASE}.{gold_table_name}"
    )

    print(f"\nValidating Gold: {gold_table_name}")

    # Check 1: Table existence

    if not table_exists(full_table_name):

        add_check(
            "GOLD",
            full_table_name,
            "Table existence",
            "MISSING",
            "AVAILABLE",
            False,
            "Gold table does not exist."
        )

        continue

    add_check(
        "GOLD",
        full_table_name,
        "Table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Gold table exists."
    )

    # Check 2: Record count

    gold_count = get_table_count(full_table_name)

    add_check(
        "GOLD",
        full_table_name,
        "Record count",
        gold_count,
        "> 0",
        gold_count > 0,
        "Gold table must contain records."
    )

    print("Gold records:", gold_count)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 13. SILVER PROCESSING REPORT VALIDATION

# COMMAND ----------

print("=" * 80)
print("SILVER PROCESSING REPORT VALIDATION")
print("=" * 80)

if not table_exists(PROCESSING_REPORT_TABLE):

    add_check(
        "SILVER_PROCESSING",
        PROCESSING_REPORT_TABLE,
        "Report table existence",
        "MISSING",
        "AVAILABLE",
        False,
        "Silver processing report does not exist."
    )

else:

    processing_df = spark.table(PROCESSING_REPORT_TABLE)

    processing_count = processing_df.count()

    add_check(
        "SILVER_PROCESSING",
        PROCESSING_REPORT_TABLE,
        "Report table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Silver processing report exists."
    )

    add_check(
        "SILVER_PROCESSING",
        PROCESSING_REPORT_TABLE,
        "Report record count",
        processing_count,
        "> 0",
        processing_count > 0,
        "Processing report must contain execution records."
    )

    # Check the latest report batch when a timestamp is available.

    latest_processing_df, processing_timestamp_column, processing_timestamp = (
        latest_batch(processing_df)
    )

    processing_batch_count = latest_processing_df.count()

    failed_processing_count, processing_status_column = (
        count_failed_records(latest_processing_df)
    )

    if failed_processing_count is None:

        add_check(
            "SILVER_PROCESSING",
            PROCESSING_REPORT_TABLE,
            "Latest processing status",
            "NO STATUS COLUMN",
            "STATUS COLUMN AVAILABLE",
            False,
            "Could not identify a processing status column."
        )

    else:

        add_check(
            "SILVER_PROCESSING",
            PROCESSING_REPORT_TABLE,
            "Latest processing failed records",
            failed_processing_count,
            0,
            failed_processing_count == 0,
            (
                f"Latest batch timestamp column: "
                f"{processing_timestamp_column}; "
                f"status column: {processing_status_column}"
            )
        )

    print("Latest processing batch records:", processing_batch_count)
    print("Latest processing timestamp:", processing_timestamp)
    print("Failed processing records:", failed_processing_count)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 14. GOLD REFRESH EXECUTION VALIDATION
# MAGIC
# MAGIC Validate only the latest Gold refresh batch.
# MAGIC Historical failed executions must not fail the latest successful run.

# COMMAND ----------

print("=" * 80)
print("GOLD REFRESH EXECUTION VALIDATION")
print("=" * 80)

if not table_exists(GOLD_REFRESH_REPORT):

    add_check(
        "GOLD_REFRESH",
        GOLD_REFRESH_REPORT,
        "Report table existence",
        "MISSING",
        "AVAILABLE",
        False,
        "Gold refresh execution report does not exist."
    )

else:

    refresh_df = spark.table(GOLD_REFRESH_REPORT)

    # Check 1: Report table existence

    add_check(
        "GOLD_REFRESH",
        GOLD_REFRESH_REPORT,
        "Report table existence",
        "AVAILABLE",
        "AVAILABLE",
        True,
        "Gold refresh execution report exists."
    )

    # Check 2: Report records

    total_refresh_records = refresh_df.count()

    add_check(
        "GOLD_REFRESH",
        GOLD_REFRESH_REPORT,
        "Report record count",
        total_refresh_records,
        "> 0",
        total_refresh_records > 0,
        "Gold refresh report must contain execution records."
    )

    # IMPORTANT FIX:
    # Only check the latest report_timestamp batch.
    # Do not count old failed execution records.

    if "report_timestamp" not in refresh_df.columns:

        add_check(
            "GOLD_REFRESH",
            GOLD_REFRESH_REPORT,
            "Latest refresh batch",
            "report_timestamp column missing",
            "report_timestamp available",
            False,
            "Cannot identify the latest refresh batch."
        )

    else:

        latest_refresh_timestamp = (
            refresh_df
            .agg(
                F.max("report_timestamp").alias("latest_timestamp")
            )
            .first()["latest_timestamp"]
        )

        latest_refresh_df = refresh_df.filter(
            F.col("report_timestamp") == F.lit(latest_refresh_timestamp)
        )

        latest_refresh_count = latest_refresh_df.count()

        # Check 3: Failed tasks in latest batch

        if "execution_status" not in latest_refresh_df.columns:

            add_check(
                "GOLD_REFRESH",
                GOLD_REFRESH_REPORT,
                "Latest refresh failed tasks",
                "execution_status column missing",
                "execution_status available",
                False,
                "Cannot validate Gold refresh execution status."
            )

        else:

            failed_refresh_count = (
                latest_refresh_df
                .filter(
                    F.upper(F.col("execution_status")) == "FAILED"
                )
                .count()
            )

            successful_refresh_count = (
                latest_refresh_df
                .filter(
                    F.upper(F.col("execution_status")) == "SUCCESS"
                )
                .count()
            )

            add_check(
                "GOLD_REFRESH",
                GOLD_REFRESH_REPORT,
                "Failed refresh tasks",
                failed_refresh_count,
                0,
                (
                    latest_refresh_count > 0
                    and failed_refresh_count == 0
                    and successful_refresh_count == latest_refresh_count
                ),
                (
                    f"Latest batch timestamp: {latest_refresh_timestamp}; "
                    f"total tasks: {latest_refresh_count}; "
                    f"successful tasks: {successful_refresh_count}; "
                    f"failed tasks: {failed_refresh_count}"
                )
            )

        print("Latest refresh timestamp:", latest_refresh_timestamp)
        print("Latest refresh tasks:", latest_refresh_count)

        if "execution_status" in latest_refresh_df.columns:

            latest_refresh_df.groupBy(
                "execution_status"
            ).count().show()


# COMMAND ----------

# MAGIC %md
# MAGIC ## 15. GENERATE VALIDATION REPORT

# COMMAND ----------

print("=" * 80)
print("GENERATING INCREMENTAL PIPELINE VALIDATION REPORT")
print("=" * 80)

validation_schema = StructType([

    StructField(
        "validation_category",
        StringType(),
        True
    ),

    StructField(
        "object_name",
        StringType(),
        True
    ),

    StructField(
        "check_name",
        StringType(),
        True
    ),

    StructField(
        "actual_value",
        StringType(),
        True
    ),

    StructField(
        "expected_value",
        StringType(),
        True
    ),

    StructField(
        "validation_status",
        StringType(),
        True
    ),

    StructField(
        "details",
        StringType(),
        True
    ),

    StructField(
        "validation_timestamp",
        TimestampType(),
        True
    )

])


validation_rows = [

    (
        result["validation_category"],
        result["object_name"],
        result["check_name"],
        result["actual_value"],
        result["expected_value"],
        result["validation_status"],
        result["details"],
        result["validation_timestamp"]
    )

    for result in validation_results

]


validation_df = spark.createDataFrame(
    validation_rows,
    schema=validation_schema
)


# COMMAND ----------

# ============================================================
# 16. VALIDATION SUMMARY
# ============================================================

total_checks = validation_df.count()

passed_checks = (
    validation_df
    .filter(F.col("validation_status") == "PASS")
    .count()
)

failed_checks = (
    validation_df
    .filter(F.col("validation_status") == "FAIL")
    .count()
)

pass_rate = (
    round((passed_checks / total_checks) * 100, 2)
    if total_checks > 0
    else 0
)

overall_status = (
    "SUCCESS"
    if failed_checks == 0 and total_checks > 0
    else "FAILED"
)

print("\n")
print("=" * 80)
print("INCREMENTAL PIPELINE VALIDATION SUMMARY")
print("=" * 80)

print(f"Total checks       : {total_checks}")
print(f"Passed checks      : {passed_checks}")
print(f"Failed checks      : {failed_checks}")
print(f"Pass rate          : {pass_rate}%")
print(f"Overall status     : {overall_status}")

print("=" * 80)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 17. DISPLAY VALIDATION DETAILS

# COMMAND ----------

print("INCREMENTAL PIPELINE VALIDATION DETAILS")

validation_df.orderBy(
    "validation_category",
    "object_name",
    "check_name"
).show(
    200,
    truncate=False
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 18. DISPLAY FAILED CHECKS

# COMMAND ----------

failed_validation_df = validation_df.filter(
    F.col("validation_status") == "FAIL"
)

print("FAILED VALIDATION CHECKS")

if failed_checks > 0:

    failed_validation_df.show(
        200,
        truncate=False
    )

else:

    print("No failed validation checks.")


# COMMAND ----------

# MAGIC %md
# MAGIC ## 19. SAVE VALIDATION REPORT

# COMMAND ----------

print("=" * 80)
print("SAVING VALIDATION REPORT")
print("=" * 80)

if not table_exists(VALIDATION_REPORT_TABLE):

    (
        validation_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(VALIDATION_REPORT_TABLE)
    )

else:

    (
        validation_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(VALIDATION_REPORT_TABLE)
    )

print("Validation report saved successfully.")

print("Report table:", VALIDATION_REPORT_TABLE)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 20. FINAL INCREMENTAL PIPELINE STATUS

# COMMAND ----------

print("\n")
print("=" * 80)
print("FINAL INCREMENTAL PIPELINE VALIDATION")
print("=" * 80)

print(f"Total checks       : {total_checks}")
print(f"Passed checks      : {passed_checks}")
print(f"Failed checks      : {failed_checks}")
print(f"Pass rate          : {pass_rate}%")

print(f"Overall pipeline status: {overall_status}")

print("Validation report:", VALIDATION_REPORT_TABLE)

print("=" * 80)


# COMMAND ----------

# ============================================================
# 21. FAIL NOTEBOOK IF VALIDATION FAILS
# ============================================================

if failed_checks > 0:

    raise RuntimeError(
        f"Incremental pipeline validation failed. "
        f"Total checks: {total_checks}, "
        f"Passed: {passed_checks}, "
        f"Failed: {failed_checks}. "
        f"Review {VALIDATION_REPORT_TABLE}."
    )

else:

    print("\n")
    print("=" * 80)
    print("INCREMENTAL PIPELINE VALIDATION COMPLETED SUCCESSFULLY")
    print("=" * 80)

    print("All validation checks passed.")
    print("Bronze validation: PASSED")
    print("Silver validation: PASSED")
    print("Ledger validation: PASSED")
    print("Quarantine validation: PASSED")
    print("Delta MERGE audit validation: PASSED")
    print("Gold validation: PASSED")
    print("Silver processing report validation: PASSED")
    print("Gold refresh validation: PASSED")

    print("=" * 80)



### Detailed validation report

In [0]:
%sql

SELECT
    validation_category,
    object_name,
    check_name,
    actual_value,
    expected_value,
    validation_status,
    details
FROM workspace.ecommerce_gold.incremental_pipeline_validation_report
ORDER BY validation_timestamp DESC;


### Overall pass/fail counts

In [0]:
%sql

SELECT
    validation_category,
    COUNT(*) AS total_checks,
    SUM(
        CASE WHEN validation_status = 'PASS'
        THEN 1 ELSE 0 END
    ) AS passed_checks,
    SUM(
        CASE WHEN validation_status = 'FAIL'
        THEN 1 ELSE 0 END
    ) AS failed_checks
FROM workspace.ecommerce_gold.incremental_pipeline_validation_report
GROUP BY validation_category
ORDER BY validation_category;


### C. Find all failed checks

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_gold.incremental_pipeline_validation_report
WHERE validation_status = 'FAIL'
ORDER BY validation_timestamp DESC;